In [1]:
import sys
print(sys.executable)

c:\T01\Agent01\.venv\Scripts\python.exe


In [113]:
from dotenv import load_dotenv
import os
from openai import OpenAI
from IPython.display import Markdown, display, HTML
import json
from langchain_community.utilities import GoogleSerperAPIWrapper
import ipywidgets as widgets
import re
import uuid

In [114]:
load_dotenv(override=True)
# openai=OpenAI()

True

In [115]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
os.getenv('GOOGLE_API_KEY')

openai = OpenAI(api_key=openai_api_key)

In [116]:
# OLLAMA_BASE_URL = "http://localhost:11434/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

In [117]:
# ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
gemini = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)

In [118]:
serper = GoogleSerperAPIWrapper()
serper_images = GoogleSerperAPIWrapper(type="images")

In [119]:
 
# request = f"""
# You are a content agent writing on the recent trends in the Technology. Your job:
 
# Write a ~500-word blog post on: "AI Agents"
#    - Tone: practical, friendly, no fluff
#    - Include a short intro, 5 numbered sections with subheads, and a 2-sentence conclusion
#    - Target audience: engineers
 
# """
# messages = [{"role": "user", "content": request}]

In [120]:
# messages

In [121]:
prompt = """
You are a content agent responsible for producing and publishing content 
for our platform. Given a category and subtopic, follow this process 
IN ORDER, using the tools available to you:

1. RESEARCH
   Call web_search tool with a short, specific query (4-6 words) to gather 
   current, factual context on the subtopic within the category. This is 
   to avoid generic or outdated filler — do not skip this step.
   You may call it up to 2 times if the first results are too broad or irrelevant.

2. WRITE CONTENT
   Using the research, write a piece with:
   - title (SEO-friendly, max 70 characters)
   - intro (2-3 sentences)
   - sections (5, each with a heading and body text)
   - conclusion (2-3 sentences)
   - tags (3-5 relevant SEO tags)
   Do not fabricate facts, statistics, or quotes not supported by the research.
   Word count target: word_count words total.

3. SOURCE IMAGES
   Call image_search tool once per needed image (3-5 total):
   - 1 hero/featured image — landscape orientation
   - 2-4 supporting images, one per relevant section - adjust accordingly that it do not end up taking more space the text content
   Only use royalty-free sources. Return the image URL and source name for each.
   If no suitable image is found for a section, skip it rather than 
   inventing a URL.
   - return the URL of each image used

4. ASSEMBLE DRAFT
   Combine the written content and images into a single JSON object 
   matching this structure:
   {
     "title": "", "slug": "", "category": "", "tags": [],
     "meta_description": "", "intro": "",
     "sections": [{"heading": "", "text": "", "image": {"url": "", "source": ""}}],
     "conclusion": "", "featured_image": {"url": "", "source": ""},
     "status": "draft"
   }

5. STOP FOR HUMAN APPROVAL
   Do NOT call publish tool yet. Present the assembled draft as your final 
   response for this turn, clearly labeled, and wait for explicit approval 
   before publishing.

6. PUBLISH (only after approval is given )
   Call publish tool with the approved payload. Set "status" to "draft" or 
   "live" based on what the human specifies.

7. REPORT
   After publish tool returns, report back the URL/ID and a 1-line summary.

Rules:
- Follow the steps in order — do not skip research or jump straight to writing.
- Never call publish tool without explicit human approval in the conversation.
- If any tool call fails, report the error and stop — do not retry more than once.
- Do not fabricate image URLs, facts, or statistics.
"""

In [122]:
def web_search(query: str)->str:
    """ Search the web for the current information on a given query"""
    return serper.run(query)

In [123]:
web_search_json = {
    "name": "web_search",
    "description": "Search the web for current information relevant to the topic",
    "parameters":{ 
        "type": "object",
        "properties":{
            "query":{
                "type": "string",
                "description": "A short, specific search query to seach the web(4-6 words)"
            } 
        },
    "required": ["query"],
    "additionalProperties": False
    }
}

In [124]:
def image_search(query):
    """Search for royalty-free images relevant to the query."""
    results = serper_images.results(query) 

    images = results.get("images", [])[:5]  
    if not images:
        return []

    return [
        {"url": img.get("imageUrl"), "source": img.get("source", "Unknown")}
        for img in images
    ]

In [125]:
image_search_json={
    "name": "image_search",
    "description": "Search the web for royalty free images on the platforms like unsplash, pixabay,p exels, etc relevant to the query",
    "parameters":{
        "type": "object",
        "properties":{
            "query":{
                "type": "string",
                "description": "A short, specific search query to search for images(4-6 words)"
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [126]:
def publish(payload):
    """Publish the content draft to the platform."""
    return payload

In [127]:
publish_json={
    "name": "publish",
    "description": "Publish the finished content draft to the platform.",
    "parameters":{
        "type": "object",
        "properties":{
            "payload":{
                "type": "string",
                "description": "The final content draft + images to publish"
            }
        },
        "required": ["payload"],
        "additionalProperties": False
    }
}

In [128]:
tools = [{"type": "function", "function": web_search_json}, {"type": "function", "function": image_search_json}, {"type": "function", "function": publish_json}]

In [129]:
tools_flow = {
    "web_search": web_search,
    "image_search": image_search,
    "publish": publish,
}                         

In [130]:
def agent01(category, subtopic, word_count):
    filled_prompt = prompt.replace("word_count", str(word_count))  # or however you're filling it
    messages = [
        {"role": "system", "content": filled_prompt},
        {"role": "user", "content": f"category: {category}, subtopic: {subtopic}"}
    ]
    response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = tools_flow[function_name](**args)
            messages.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
        response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages, tools=tools)

    return response.choices[0].message.content, messages

In [131]:
def review_draft(messages, decision, feedback=None, live=False):
    if decision == "reject":
        if not feedback:
            raise ValueError("Feedback is required when rejecting a draft.")
        user_msg = f"Not approved. Please revise the draft based on this feedback: {feedback}"
    elif decision == "approve":
        user_msg = f"Approved. Publish with status = {'live' if live else 'draft'}."
    else:
        raise ValueError("decision must be 'approve' or 'reject'")

    messages.append({"role": "user", "content": user_msg})
    response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        messages.append(message)
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = tools_flow[fn_name](**args)
            messages.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
        response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages, tools=tools)

    return response.choices[0].message.content, messages

In [132]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def show_approval_buttons(messages):
    output = widgets.Output()

    approve_btn = widgets.Button(description="Approve (Draft)", button_style="success")
    approve_live_btn = widgets.Button(description="Approve (Live)", button_style="info")
    reject_btn = widgets.Button(description="Reject", button_style="danger")

    def on_approve(b, live=False):
        with output:
            clear_output()
            print("Publishing..." if live else "Saving as draft...")
            result = review_draft(messages, live=live)
            print(result)

    def on_reject(b):
        with output:
            clear_output()
            print("Rejected. Add feedback and re-run agent01 with revised instructions.")

    approve_btn.on_click(lambda b: on_approve(b, live=False))
    approve_live_btn.on_click(lambda b: on_approve(b, live=True))
    reject_btn.on_click(on_reject)

    display(widgets.HBox([approve_btn, approve_live_btn, reject_btn]), output)

In [133]:
def clean_json_string(s):
    s = s.strip()

    # strip markdown code fences if present
    if s.startswith("```"):
        s = s.split("\n", 1)[1]
        s = s.rsplit("```", 1)[0]
        s = s.strip()

    # extract only the first complete {...} block, ignore any trailing text
    match = re.search(r'\{.*\}', s, re.DOTALL)
    if match:
        s = match.group(0)

    return s.strip()

In [134]:

def display_draft(draft_json_str):
    draft_json_str = clean_json_string(draft_json_str)
    data = json.loads(draft_json_str)
 
    title = data.get('title') or '(no title)'
    meta_description = data.get('meta_description') or ''
    category = data.get('category') or ''
    tags = data.get('tags') or []
    intro = data.get('intro') or ''
    sections = data.get('sections') or []
    conclusion = data.get('conclusion') or ''
    status = data.get('status') or 'unknown'
    featured = data.get('featured_image') or {}
 
    # collect all images: featured first, then any section images, dedup + skip empty
    all_images = []
    if featured.get('url'):
        all_images.append(featured['url'])
    for section in sections:
        img = section.get('image') or {}
        url = img.get('url')
        if url and url not in all_images:
            all_images.append(url)
 
    gallery_id = f"gallery_{uuid.uuid4().hex[:8]}"
 
    # --- Build gallery HTML (hero + vertical thumbnail strip) ---
    if all_images:
        thumbs_html = "".join(
            f'<img src="{url}" class="{gallery_id}-thumb" '
            f'onclick="document.getElementById(\'{gallery_id}-hero\').src=\'{url}\'; '
            f'document.querySelectorAll(\'.{gallery_id}-thumb\').forEach(t=>t.classList.remove(\'active\')); '
            f'this.classList.add(\'active\')" />'
            for url in all_images
        )
        gallery_html = f"""
        <div style="display:flex; gap:12px; margin:16px 0;">
            <div style="flex:1; max-width:600px;">
                <img id="{gallery_id}-hero" src="{all_images[0]}"
                     style="width:100%; border-radius:8px; object-fit:cover; aspect-ratio:16/9;" />
            </div>
            <div style="display:flex; flex-direction:column; gap:8px; width:90px;">
                {thumbs_html}
            </div>
        </div>
        <style>
            .{gallery_id}-thumb {{
                width:100%; height:60px; object-fit:cover; border-radius:6px;
                cursor:pointer; opacity:0.65; border:2px solid transparent;
                transition: all 0.15s ease;
            }}
            .{gallery_id}-thumb:hover {{ opacity:1; }}
            .{gallery_id}-thumb.active {{ opacity:1; border-color:#4a90d9; }}
        </style>
        """
    else:
        gallery_html = "<p><em>(no images sourced for this draft)</em></p>"
 
    # --- Build sections HTML (text only now, no per-section image) ---
    sections_html = ""
    for section in sections:
        heading = section.get('heading') or ''
        text = section.get('text') or ''
        sections_html += f"<h3>{heading}</h3><p>{text}</p>"
 
    html = f"""
    <div style="font-family:sans-serif; max-width:700px; line-height:1.5;">
        <h1>{title}</h1>
        <p><em>{meta_description}</em></p>
        <p><strong>Category:</strong> {category} | <strong>Tags:</strong> {', '.join(tags)}</p>
 
        {gallery_html}
 
        <p>{intro}</p>
        {sections_html}
        <p>{conclusion}</p>
        <hr/>
        <p><strong>Status:</strong> {status}</p>
    </div>
    """
 
    display(HTML(html))


In [144]:
draft, messages = agent01(category="Business", subtopic="Finance", word_count=600)
display_draft(draft)

# print(repr(draft))


In [146]:
revised_draft, messages = review_draft(messages, decision="reject", feedback="find new more detail-oriented and the subtopic oriented images.")
# print(repr(revised_draft))
display_draft(revised_draft) 


In [ ]:
result, messages = review_draft(messages, decision="approve", live=False)
print(result)

The content has been successfully published as a draft. You can access it using the following details:

*   **Status:** Draft
*   **Summary:** "Core Principles of Effective Business Financial Management" has been drafted, covering key areas such as cash flow, strategic budgeting, risk assessment, time value of money, and financial accountability.


In [138]:
# !ollama pull llama3.2
 
# model_name = "llama3.2"
# model_name = "gpt-5.4-mini"
# model_name = "gemini-3.1-flash-lite"
 

In [139]:
# response = ollama.chat.completions.create(model=model_name, messages=messages)
# response = openai.chat.completions.create(model=model_name, messages=messages)
# response = gemini.chat.completions.create(model=model_name, messages=messages)
# answer = response.choices[0].message.content

# print(answer)

In [140]:
# display(Markdown(answer))